# EDA — TipoDecisao

Exploratory Data Analysis using `TipoDecisao` as the target variable (document type classification).

In [ ]:
import json
import pandas as pd
from pathlib import Path
import re

DATA_DIR = Path('../../data/1_raw')
PROJECT_ROOT = Path('../..').resolve()


In [ ]:
def field_info(df, n=3):
    rows = []
    for col in df.columns:
        samples = df[col].dropna().head(n).tolist()
        samples = [str(v)[:60] + '...' if isinstance(v, str) and len(v) > 60 else v for v in samples]
        rows.append({'field': col, 'dtype': str(df[col].dtype), 'examples': samples})
    return pd.DataFrame(rows)


## 1. Data Loading

### 1.1 Load the Datasets

#### STJ (Supremo Tribunal da Justiça)

In [ ]:
with open(DATA_DIR / 'STJ.json') as f:
    stj = pd.DataFrame(json.load(f))

print(f'Records : {stj.shape[0]}')
print(f'Fields  : {stj.shape[1]}')
stj.head(2)


In [ ]:
field_info(stj)


#### STA (Supremo Tribunal Administrativo)

In [ ]:
with open(DATA_DIR / 'STA.json') as f:
    sta = pd.DataFrame(json.load(f))

print(f'Records : {sta.shape[0]}')
print(f'Fields  : {sta.shape[1]}')
sta.head(2)


In [ ]:
field_info(sta)


#### TCAN (Tribunal Central Administrativo Norte)

In [ ]:
with open(DATA_DIR / 'TCAN.json') as f:
    tcan = pd.DataFrame(json.load(f))

print(f'Records : {tcan.shape[0]}')
print(f'Fields  : {tcan.shape[1]}')
tcan.head(2)


In [ ]:
field_info(tcan)


#### TCAS (Tribunal Central Administrativo Sul)

In [ ]:
with open(DATA_DIR / 'TCAS.json') as f:
    tcas = pd.DataFrame(json.load(f))

print(f'Records : {tcas.shape[0]}')
print(f'Fields  : {tcas.shape[1]}')
tcas.head(2)


In [ ]:
field_info(tcas)


#### TRE (Tribunal da Relação de Évora)

In [ ]:
with open(DATA_DIR / 'TRE.json') as f:
    tre = pd.DataFrame(json.load(f))

print(f'Records : {tre.shape[0]}')
print(f'Fields  : {tre.shape[1]}')
tre.head(2)


In [ ]:
field_info(tre)


#### TRG (Tribunal da Relação de Guimarães)

In [ ]:
with open(DATA_DIR / 'TRG.json') as f:
    trg = pd.DataFrame(json.load(f))

print(f'Records : {trg.shape[0]}')
print(f'Fields  : {trg.shape[1]}')
trg.head(2)


In [ ]:
field_info(trg)


#### TRL (Tribunal da Relação de Lisboa)

In [ ]:
with open(DATA_DIR / 'TRL.json') as f:
    trl = pd.DataFrame(json.load(f))

print(f'Records : {trl.shape[0]}')
print(f'Fields  : {trl.shape[1]}')
trl.head(2)


In [ ]:
field_info(trl)


#### TRP (Tribunal da Relação do Porto)

In [ ]:
with open(DATA_DIR / 'TRP.json') as f:
    trp = pd.DataFrame(json.load(f))

print(f'Records : {trp.shape[0]}')
print(f'Fields  : {trp.shape[1]}')
trp.head(2)


In [ ]:
field_info(trp)


#### TRC (Tribunal da Relação de Coimbra)

In [ ]:
with open(DATA_DIR / 'TRC.json') as f:
    trc = pd.DataFrame(json.load(f))

print(f'Records : {trc.shape[0]}')
print(f'Fields  : {trc.shape[1]}')
trc.head(2)


In [ ]:
field_info(trc)


#### TConflitos (Tribunal de Conflitos)

In [ ]:
with open(DATA_DIR / 'TConflitos.json') as f:
    tconflitos = pd.DataFrame(json.load(f))

print(f'Records : {tconflitos.shape[0]}')
print(f'Fields  : {tconflitos.shape[1]}')
tconflitos.head(2)


In [ ]:
field_info(tconflitos)


#### Global Table

In [ ]:
summary = pd.DataFrame([
    {'source': 'STJ',        'records': stj.shape[0],        'fields': stj.shape[1]},
    {'source': 'STA',        'records': sta.shape[0],        'fields': sta.shape[1]},
    {'source': 'TCAN',       'records': tcan.shape[0],       'fields': tcan.shape[1]},
    {'source': 'TCAS',       'records': tcas.shape[0],       'fields': tcas.shape[1]},
    {'source': 'TRE',        'records': tre.shape[0],        'fields': tre.shape[1]},
    {'source': 'TRG',        'records': trg.shape[0],        'fields': trg.shape[1]},
    {'source': 'TRL',        'records': trl.shape[0],        'fields': trl.shape[1]},
    {'source': 'TRP',        'records': trp.shape[0],        'fields': trp.shape[1]},
    {'source': 'TRC',        'records': trc.shape[0],        'fields': trc.shape[1]},
    {'source': 'TConflitos', 'records': tconflitos.shape[0], 'fields': tconflitos.shape[1]},
])
summary


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(summary['source'], summary['records'], color='steelblue')
ax.set_xlabel('Source')
ax.set_ylabel('Records')
ax.set_title('Records per tribunal')
for bar, val in zip(bars, summary['records']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(val), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


### 1.2 Combined Dataset

In [ ]:
all_data = pd.concat([
    stj.assign(source='STJ'),
    sta.assign(source='STA'),
    tcan.assign(source='TCAN'),
    tcas.assign(source='TCAS'),
    tre.assign(source='TRE'),
    trg.assign(source='TRG'),
    trl.assign(source='TRL'),
    trp.assign(source='TRP'),
    trc.assign(source='TRC'),
    tconflitos.assign(source='TConflitos'),
], ignore_index=True)

print(f'Total records : {len(all_data):,}')
print(f'Sources       : {all_data["source"].nunique()}')


## 2. Data Quality Assessment

In [ ]:
total = len(all_data)

missing = pd.DataFrame({
    'null_count':   all_data.isnull().sum(),
    'empty_count':  (all_data == '').sum(),
}).assign(
    total_missing=lambda df: df['null_count'] + df['empty_count'],
    pct=lambda df: (df['total_missing'] / total * 100).round(2),
).sort_values('total_missing', ascending=False)

missing[missing['total_missing'] > 0]


#### Sumario — missing breakdown by source

In [ ]:
sumario_missing = (
    all_data.assign(missing=all_data['Sumario'].isna() | (all_data['Sumario'] == ''))
    .groupby('source')
    .agg(total=('missing', 'count'), missing=('missing', 'sum'))
    .assign(pct=lambda df: (df['missing'] / df['total'] * 100).round(2))
    .sort_values('missing', ascending=False)
)
sumario_missing


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = sumario_missing.index
width = 0.4

ax.bar(x, sumario_missing['total'],   width, label='Total', color='steelblue')
ax.bar(x, sumario_missing['missing'], width, label='Sumario em falta', color='tomato')

for bar, pct in zip(ax.patches[len(x):], sumario_missing['pct']):
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                f'{pct}%', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos')
ax.set_title('Sumario em falta por tribunal')
ax.legend()
plt.tight_layout()
plt.show()


#### Sumario — Global Overview

In [ ]:
sumario = all_data['Sumario']
print(f'Null values   : {sumario.isna().sum()}')
print(f'Empty strings : {(sumario == "").sum()}')
print()
sumario.str.len().describe().rename('char_length')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(all_data['Sumario'].str.len().dropna(), vert=False, flierprops=dict(marker='.', alpha=0.3))
ax.set_xlabel('Nº de caracteres')
ax.set_title('Distribuição do comprimento do Sumário')
plt.tight_layout()
plt.show()


In [ ]:
import plotly.express as px

lengths_filtered = all_data['Sumario'].str.len().dropna()
lengths_filtered = lengths_filtered[lengths_filtered < 500_000]

fig = px.box(x=lengths_filtered, labels={'x': 'Nº de caracteres'},
             title='Distribuição do comprimento do Sumário (excl. outlier 500k)')
fig.show()


In [ ]:
(
    all_data.groupby('source')['Sumario']
    .agg(
        nulls=lambda s: s.isna().sum(),
        empty=lambda s: (s == '').sum(),
        count='count',
        mean=lambda s: s.str.len().mean(),
        std=lambda s: s.str.len().std(),
        min=lambda s: s.str.len().min(),
        median=lambda s: s.str.len().median(),
        max=lambda s: s.str.len().max(),
    )
    .round(1)
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sumario.str.len().dropna().pipe(lambda s: s[s < 500_000]).plot(
    kind='hist', bins=50, ax=ax, color='steelblue', edgecolor='white'
)
ax.set_xlabel('Character length')
ax.set_ylabel('Count')
ax.set_title('Sumario length distribution (excl. outlier 500k)')
plt.tight_layout()
plt.show()


- Documento com o máximo de carateres.

In [ ]:
idx = all_data['Sumario'].str.len().idxmax()
doc = all_data.loc[idx]

output_path = PROJECT_ROOT / 'sumario_max.txt'
output_path.write_text(doc['Sumario'], encoding='utf-8')

print(f'Processo : {doc["NumeroProcesso"]}')
print(f'Guardado em: {output_path}')
print(f'Tamanho: {len(doc["Sumario"]):,} caracteres')

Checking manually the null values.

In [ ]:
null_idx = all_data[all_data['Sumario'].isna()].index.tolist()
print(f'Null indices ({len(null_idx)} total): {null_idx[:20]} ...')

if null_idx:
    i = null_idx[0]
    all_data.loc[i, ['source', 'OrigemEnum', 'NumeroProcesso', 'TipoDecisao', 'Sumario', 'Texto']].to_frame('value')


In [ ]:
out_dir = Path('../../null_values')
out_dir.mkdir(exist_ok=True)

if null_idx:
    i = null_idx[0]
    row = all_data.loc[i]
    out_path = out_dir / f'tipodecisao_idx_{i}.txt'
    with open(out_path, 'w', encoding='utf-8') as f:
        for col in all_data.columns:
            f.write(f'=== {col} ===\n{row[col]}\n\n')
    print(f'Saved to {out_path.resolve()}')
else:
    print('No null Sumario values.')


### 2.2 Duplicate Records

Two levels of duplication are checked.

In [ ]:
dup_process = all_data[all_data.duplicated('NumeroProcesso', keep=False)].shape[0]
dup_sumario = all_data[all_data['Sumario'].notna()
                       & all_data.duplicated('Sumario', keep=False)].shape[0]

def normalise(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sumario_norm = all_data['Sumario'].dropna().apply(normalise)
near_dup_mask = sumario_norm.duplicated(keep=False)
dup_near = near_dup_mask.sum()

pd.DataFrame({
    'check': [
        'Same NumeroProcesso',
        'Same Sumario (exact)',
        'Same Sumario (normalised)',
    ],
    'affected_records': [dup_process, dup_sumario, dup_near],
    'pct': [
        round(dup_process / total * 100, 2),
        round(dup_sumario / total * 100, 2),
        round(dup_near    / total * 100, 2),
    ],
})


In [ ]:
all_data[all_data['Sumario'].notna() & all_data.duplicated('Sumario', keep=False)] \
    .sort_values('Sumario') \
    .head(5)[['NumeroProcesso', 'Origem', 'Data', 'Sumario']] \
    .assign(Sumario=lambda df: df['Sumario'].str[:200])

In [ ]:
exact_dups = all_data[all_data['Sumario'].notna() & all_data.duplicated('Sumario', keep=False)] \
    .sort_values('Sumario')

output_path = PROJECT_ROOT / 'exact_sumario_dups.txt'
with output_path.open('w', encoding='utf-8') as f:
    for i, (_, row) in enumerate(exact_dups.head(15).iterrows(), 1):
        f.write(f"=== Exemplo {i} ===\n")
        f.write(f"Processo : {row['NumeroProcesso']}\n")
        f.write(f"Origem   : {row['Origem']}\n")
        f.write(f"Data     : {row['Data']}\n")
        f.write(f"Sumário  :\n{row['Sumario']}\n\n")

print(f'Guardado em: {output_path}')


In [ ]:
norm_dups = all_data[all_data['Sumario'].notna()].copy()
norm_dups['Sumario_norm'] = norm_dups['Sumario'].apply(normalise)
norm_dups = norm_dups[norm_dups.duplicated('Sumario_norm', keep=False)] \
    .sort_values('Sumario_norm')

output_path = PROJECT_ROOT / 'near_sumario_dups.txt'
with output_path.open('w', encoding='utf-8') as f:
    for i, (_, row) in enumerate(norm_dups.head(15).iterrows(), 1):
        f.write(f"=== Exemplo {i} ===\n")
        f.write(f"Processo : {row['NumeroProcesso']}\n")
        f.write(f"Origem   : {row['Origem']}\n")
        f.write(f"Data     : {row['Data']}\n")
        f.write(f"Sumário  :\n{row['Sumario']}\n\n")

print(f'Guardado em: {output_path}')


In [ ]:
near_dup_rows = all_data.loc[sumario_norm[near_dup_mask].index,
                             ['source', 'NumeroProcesso', 'TipoDecisao', 'Sumario']].copy()
near_dup_rows['sumario_norm'] = sumario_norm[near_dup_mask]

print(f'Grupos de quasi-duplicados: {near_dup_rows.groupby("sumario_norm").ngroups}')
near_dup_rows.groupby('source').size().rename('quasi_dup_count').to_frame()


In [ ]:
exact_by_source = (
    all_data[all_data['Sumario'].notna() & all_data.duplicated('Sumario', keep=False)]
    .groupby('source').size().rename('exato')
)
near_by_source = near_dup_rows.groupby('source').size().rename('normalizado')

dup_by_source = pd.concat([exact_by_source, near_by_source], axis=1).fillna(0).astype(int)
dup_by_source = dup_by_source.reindex(all_data['source'].unique()).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(dup_by_source))
w = 0.35
ax.bar([i - w/2 for i in x], dup_by_source['exato'],      w, color='steelblue',
       label=f'Duplicado exato ({dup_by_source["exato"].sum()} / {total})')
ax.bar([i + w/2 for i in x], dup_by_source['normalizado'], w, color='tomato',
       label=f'Quase duplicado ({dup_by_source["normalizado"].sum()} / {total})')
ax.set_xticks(list(x))
ax.set_xticklabels(dup_by_source.index)
ax.set_xlabel('Tribunal')
ax.set_ylabel('Registos afectados')
ax.set_title('Duplicados por tribunal (Sumario)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
exact_by_tipo = (
    all_data[all_data['Sumario'].notna() & all_data.duplicated('Sumario', keep=False)]
    .groupby('TipoDecisao').size().rename('exacto')
)
near_by_tipo = near_dup_rows.groupby('TipoDecisao').size().rename('normalizado')

dup_by_tipo = pd.concat([exact_by_tipo, near_by_tipo], axis=1).fillna(0).astype(int)
dup_by_tipo = dup_by_tipo.reindex(all_data['TipoDecisao'].unique()).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(dup_by_tipo))
w = 0.35
bars_exact = ax.bar([i - w/2 for i in x], dup_by_tipo['exacto'],      w, color='steelblue',
                    label=f'Duplicado exacto ({dup_by_tipo["exacto"].sum()} / {total})')
bars_near  = ax.bar([i + w/2 for i in x], dup_by_tipo['normalizado'], w, color='tomato',
                    label=f'Quasi-duplicado ({dup_by_tipo["normalizado"].sum()} / {total})')

for bar in bars_exact:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
for bar in bars_near:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(dup_by_tipo.index, rotation=15, ha='right')
ax.set_xlabel('TipoDecisao')
ax.set_ylabel('Registos afectados')
ax.set_title('Duplicados por tipo de documento (Sumario)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3 Entradas Malformadas ou Placeholder no Sumario

In [ ]:
PLACEHOLDER_PATTERNS = [
    (r'^\s*$',                          'só espaços'),
    (r'^\s*[.\-_/\\|]{1,3}\s*$',    'só pontuação'),
    (r'^\s*n\.?/?a\.?\s*$',          'N/A'),
    (r'^\s*na[oã]?\s+h[aá]\s*',      'sem sumário (texto)'),
    (r'^\s*sem\s+sum[aá]rio\s*$',     'sem sumário'),
    (r'^\s*.{1,3}\s*$',               '<= 3 caracteres'),
    (r'^\s*.{4,9}\s*$',               '4-9 caracteres'),
]

s = all_data['Sumario'].dropna()
results = []
for pattern, label in PLACEHOLDER_PATTERNS:
    mask = s.str.match(pattern, case=False)
    results.append({
        'categoria':  label,
        'padrao':     pattern,
        'registos':   mask.sum(),
        'pct':        round(mask.sum() / total * 100, 3),
    })

pd.DataFrame(results)


#### Sumarios suspeitos — menos de 20 tokens

In [ ]:
TOKEN_THRESHOLD = 20

token_counts = all_data['Sumario'].dropna().str.split().str.len()
short_flag = token_counts < TOKEN_THRESHOLD

flagged = all_data.loc[short_flag[short_flag].index,
                       ['source', 'TipoDecisao', 'NumeroProcesso', 'Sumario']].copy()
flagged['token_count'] = token_counts[short_flag].values

print(f'Registos com Sumario presente mas < {TOKEN_THRESHOLD} tokens : {len(flagged)} ({len(flagged)/total*100:.2f}%)')
print()
flagged.groupby('source')['token_count'].agg(['count', 'min', 'mean', 'max']).round(1)


In [ ]:
flagged.sort_values('token_count').head(10)[['source', 'TipoDecisao', 'NumeroProcesso', 'token_count', 'Sumario']]


In [ ]:
out_dir = Path('../../short_sumario')
out_dir.mkdir(exist_ok=True)

examples = flagged.sort_values('token_count').head(20)
for idx, row in examples.iterrows():
    out_path = out_dir / f'tipodecisao_idx_{idx}.txt'
    with open(out_path, 'w', encoding='utf-8') as f:
        for col in all_data.columns:
            f.write(f'=== {col} ===\n{all_data.loc[idx, col]}\n\n')

print(f'Guardados {len(examples)} exemplos em {out_dir.resolve()}')


In [ ]:
short_by_tipo = flagged.groupby('TipoDecisao').size()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(short_by_tipo.index, short_by_tipo.values, color='steelblue')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('TipoDecisao')
ax.set_ylabel('Registos')
ax.set_title(f'Sumarios com menos de {TOKEN_THRESHOLD} tokens por tipo de documento')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()


## 3. Target Variable Analysis

In [ ]:
class_counts = all_data['TipoDecisao'].value_counts()
n_classes = class_counts.shape[0]

print(f'Numero de classes (TipoDecisao) : {n_classes}')
print(f'Total de registos               : {len(all_data):,}')
print(f'Registos sem TipoDecisao        : {all_data["TipoDecisao"].isna().sum()}')
print()
class_counts.to_frame('registos').assign(
    pct=lambda df: (df['registos'] / len(all_data) * 100).round(2)
)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bars = ax1.bar(class_counts.index, class_counts.values, color='steelblue')
for bar, val in zip(bars, class_counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
             str(val), ha='center', va='bottom', fontsize=9)
ax1.set_xlabel('TipoDecisao')
ax1.set_ylabel('Registos')
ax1.set_title('Distribuicao absoluta')
ax1.tick_params(axis='x', rotation=15)

pcts = class_counts / len(all_data) * 100
bars2 = ax2.bar(pcts.index, pcts.values, color='steelblue')
for bar, val in zip(bars2, pcts.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax2.set_xlabel('TipoDecisao')
ax2.set_ylabel('Percentagem (%)')
ax2.set_title('Distribuicao relativa')
ax2.tick_params(axis='x', rotation=15)

plt.suptitle('Distribuicao da variavel alvo — TipoDecisao', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


Cobertura total.

#### Distribuição de TipoDecisao por Tribunal

In [ ]:
source_tipo = (
    all_data.groupby(['source', 'TipoDecisao'])
    .size()
    .unstack(fill_value=0)
)
source_tipo_pct = source_tipo.div(source_tipo.sum(axis=1), axis=0) * 100

print('Counts:')
display(source_tipo)
print('\nPercentages (row-normalised):')
display(source_tipo_pct.round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['steelblue', 'tomato', 'mediumseagreen']

source_tipo.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_xlabel('Tribunal')
axes[0].set_ylabel('Registos')
axes[0].set_title('TipoDecisao por tribunal (absoluto)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='TipoDecisao', fontsize=8)

source_tipo_pct.plot(kind='bar', stacked=True, ax=axes[1], color=colors, edgecolor='white')
axes[1].set_xlabel('Tribunal')
axes[1].set_ylabel('Percentagem (%)')
axes[1].set_title('TipoDecisao por tribunal (proporcional)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='TipoDecisao', fontsize=8)

plt.tight_layout()
plt.show()


#### Imbalance Summary

`TipoDecisao` has 100% coverage across all records.

In [ ]:
imbalance = class_counts.to_frame('count').assign(
    pct=lambda d: (d['count'] / len(all_data) * 100).round(1),
    imbalance_ratio=lambda d: (class_counts.max() / d['count']).round(1),
)
print('Majority-class dummy accuracy: {:.1f}%'.format(class_counts.max() / len(all_data) * 100))
print('=> Use macro-F1 as primary evaluation metric.')
print()
display(imbalance)


In [ ]:
before = len(all_data)
all_data = all_data[all_data['Sumario'].notna() & all_data['Sumario'].str.strip().ne('')].reset_index(drop=True)
after = len(all_data)

print(f'Dropped  : {before - after:,} records ({(before - after) / before * 100:.1f}%)')
print(f'Remaining: {after:,} records')
all_data['TipoDecisao'].value_counts()

## 4. Sumario Text Analysis

### 4.1 Length & Token-Count Distributions

Metrics computed on all non-null `Sumario` entries: - `char_len`.

In [ ]:
from transformers import AutoTokenizer

df_len = all_data.copy()
df_len['char_len']   = df_len['Sumario'].str.len()
df_len['word_count'] = df_len['Sumario'].str.split().str.len()

mask = df_len['Sumario'].notna()
tok = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
texts = df_len.loc[mask, 'Sumario'].tolist()
encodings = tok(texts, add_special_tokens=True, truncation=False, padding=False, return_attention_mask=False)
df_len.loc[mask, 'token_count'] = [len(ids) for ids in encodings['input_ids']]
df_len['over_512'] = df_len['token_count'] > 512

analysed = mask.sum()
print(f'Entries analysed : {analysed:,}')
print(f'Flagged >512 tok : {df_len["over_512"].sum():,}  ({df_len["over_512"].sum() / analysed * 100:.1f}%)')
print()
df_len[['char_len', 'word_count', 'token_count']].describe().round(1)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cols = [('char_len', 'Character length', None),
        ('word_count', 'Word count', None),
        ('token_count', 'Token count (BERT)', 512)]

for ax, (col, label, vline) in zip(axes, cols):
    data = df_len[col].dropna()
    data = data[data <= data.quantile(0.99)]

    ax.hist(data, bins=60, color='steelblue', edgecolor='white')
    ax.axvline(data.mean(),   color='tomato', linestyle='--', linewidth=1.2, label='mean')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.2, label='median')
    if vline:
        ax.axvline(vline, color='crimson', linestyle=':', linewidth=1.5, label=f'{vline}-token limit')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'Global distribution — {label}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
stats_global = (
    df_len[['char_len', 'word_count', 'token_count']]
    .agg(['mean', 'median', 'min', 'max', 'std'])
    .round(1)
    .rename(index={'mean': 'Mean', 'median': 'Median',
                   'min': 'Min', 'max': 'Max', 'std': 'Std'})
)
print('--- Global ---')
display(stats_global)

stats_per_class = (
    df_len.groupby('TipoDecisao')[['char_len', 'word_count', 'token_count']]
    .agg(['mean', 'median', 'min', 'max', 'std'])
    .round(1)
)
print('\n--- Per TipoDecisao class ---')
display(stats_per_class)


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

classes = df_len['TipoDecisao'].unique()
col_pairs = [('char_len', 'Character length', None),
             ('word_count', 'Word count', None),
             ('token_count', 'Token count (BERT)', 512)]

fig = make_subplots(rows=1, cols=3, subplot_titles=[label for _, label, _ in col_pairs])

for col_idx, (col, label, hline) in enumerate(col_pairs, start=1):
    for cls in classes:
        vals = df_len.loc[df_len['TipoDecisao'] == cls, col].dropna()
        fig.add_trace(
            go.Box(y=vals, name=cls, showlegend=(col_idx == 1), boxmean=False),
            row=1, col=col_idx
        )
    if hline:
        fig.add_hline(y=hline, line_color='crimson', line_dash='dot',
                      annotation_text=f'{hline}-token limit',
                      annotation_position='top right',
                      row=1, col=col_idx)
    fig.update_yaxes(title_text=label, row=1, col=col_idx)

fig.update_layout(height=500, width=1200, title_text='Length distributions by TipoDecisao',
                  boxmode='group')
fig


In [ ]:
flag_global = pd.DataFrame({
    'total': [len(df_len)],
    'over_512': [df_len['over_512'].sum()],
    'pct': [round(df_len['over_512'].mean() * 100, 2)]
}, index=['ALL'])

flag_by_source = (
    df_len.groupby('source')
    .agg(total=('over_512', 'count'), over_512=('over_512', 'sum'))
    .assign(pct=lambda d: (d['over_512'] / d['total'] * 100).round(2))
)

flag_by_class = (
    df_len.groupby('TipoDecisao')
    .agg(total=('over_512', 'count'), over_512=('over_512', 'sum'))
    .assign(pct=lambda d: (d['over_512'] / d['total'] * 100).round(2))
)

print('--- Global ---');                display(flag_global)
print('\n--- By source ---');           display(flag_by_source)
print('\n--- By TipoDecisao class ---'); display(flag_by_class)


### 4.2 Vocabulary Analysis

In [ ]:
import spacy
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

nlp = spacy.load('pt_core_news_sm', disable=['parser', 'ner', 'lemmatizer'])
PT_STOPWORDS = nlp.Defaults.stop_words

def tokenize(text):
    tokens = re.sub(r'[^\w\s]', ' ', text.lower()).split()
    return [t for t in tokens if t not in PT_STOPWORDS and len(t) > 2 and not t.isdigit()]

def tokenize_str(text):
    return ' '.join(tokenize(text))


In [ ]:
all_tokens = []
for text in df_len['Sumario']:
    all_tokens.extend(tokenize(text))

token_freq = Counter(all_tokens)

total_tok = len(all_tokens)
cumsum, coverage_vocab = 0, 0
for count in sorted(token_freq.values(), reverse=True):
    cumsum += count
    coverage_vocab += 1
    if cumsum / total_tok >= 0.80:
        break

print(f'Total tokens (stopwords removed) : {total_tok:,}')
print(f'Unique vocabulary size           : {len(token_freq):,}')
print(f'Appear once                      : {sum(1 for v in token_freq.values() if v == 1):,}')
print(f'Tokens covering 80% of corpus    : {coverage_vocab:,}')


In [ ]:
top_n = 30
terms, counts = zip(*token_freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(terms[::-1], counts[::-1], color='steelblue')
ax.set_xlabel('Frequency')
ax.set_title(f'Top {top_n} tokens — global (stopwords removed)')
plt.tight_layout()
plt.show()


In [ ]:
doc_freq = Counter()
for text in df_len['Sumario']:
    doc_freq.update(set(tokenize(text)))

top_n = 30
df_terms, df_counts = zip(*doc_freq.most_common(top_n))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_terms[::-1], df_counts[::-1], color='steelblue')
ax.set_xlabel('Document Frequency (# documents)')
ax.set_title(f'Top {top_n} tokens — global (Document Frequency, stopwords removed)')
plt.tight_layout()
plt.show()


In [ ]:
TOP_TERMS = 15
tfidf_rows = []

for cls in sorted(df_len['TipoDecisao'].unique()):
    docs = (
        df_len.loc[df_len['TipoDecisao'] == cls, 'Sumario']
        .dropna()
        .apply(tokenize_str)
        .tolist()
    )
    if len(docs) < 5:
        continue
    vec = TfidfVectorizer()
    X   = vec.fit_transform(docs)
    mean_scores = X.mean(axis=0).A1
    features    = vec.get_feature_names_out()
    for rank, idx in enumerate(mean_scores.argsort()[::-1][:TOP_TERMS], 1):
        tfidf_rows.append({'TipoDecisao': cls, 'rank': rank,
                           'term': features[idx], 'score': round(mean_scores[idx], 4)})

tfidf_df = pd.DataFrame(tfidf_rows)
tfidf_df.pivot(index='rank', columns='TipoDecisao', values='term')


In [ ]:
corpus = df_len['Sumario'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True).tolist()
labels = df_len['TipoDecisao'].tolist()

vec = CountVectorizer(ngram_range=(3, 5), min_df=0.02, max_features=100_000)
X   = vec.fit_transform(corpus)
features = vec.get_feature_names_out()

class_phrases = {}
for cls in sorted(set(labels)):
    idx     = [i for i, l in enumerate(labels) if l == cls]
    X_cls   = X[idx]
    doc_freq = (X_cls > 0).mean(axis=0).A1
    class_phrases[cls] = set(features[doc_freq >= 0.30])

boilerplate = sorted(set.intersection(*class_phrases.values()))
print(f'Boilerplate phrases (>=30% of docs in every class): {len(boilerplate)}')
print()
coverage = {cls: f'{len(class_phrases[cls]):,}' for cls in sorted(class_phrases)}
print('Phrases meeting 30% threshold per class:')
for cls, n in coverage.items():
    print(f'  {cls:<30} {n}')
print()
pd.Series(boilerplate, name='boilerplate_phrase').to_frame().head(30)


### 4.3 Linguistic Patterns

Checks language consistency, detects mixed-language or corrupted entries, and measures structural complexity of the `Sumario` field.

#### Language Detection

In [ ]:
from langdetect import detect, LangDetectException

def detect_lang(text):
    try:
        return detect(text)
    except LangDetectException:
        return 'error'

df_lang = df_len[['Sumario', 'source', 'TipoDecisao']].copy()
df_lang['lang'] = df_lang['Sumario'].apply(detect_lang)

lang_counts = df_lang['lang'].value_counts()
print('Language distribution:')
print(lang_counts.to_string())
print(f'\nPortuguese (pt): {lang_counts.get("pt", 0) / len(df_lang) * 100:.1f}% of entries')


#### Mixed-Language and Corrupted Entries

In [ ]:
non_pt = df_lang[df_lang['lang'] != 'pt'].copy()
print(f'Non-Portuguese entries: {len(non_pt)} ({len(non_pt)/len(df_lang)*100:.1f}%)')
print()

df_lang['alpha_ratio'] = df_lang['Sumario'].str.replace(r'\s', '', regex=True).apply(
    lambda s: sum(c.isalpha() for c in s) / max(len(s), 1)
)
corrupted = df_lang[df_lang['alpha_ratio'] < 0.7]
print(f'Potentially corrupted (alpha ratio < 0.70): {len(corrupted)} entries')
print()

if len(non_pt) > 0:
    display(non_pt[['source', 'TipoDecisao', 'lang', 'Sumario']].head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lang_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Detected language distribution')
axes[0].set_xlabel('Language code')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

axes[1].hist(df_lang['alpha_ratio'], bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(0.7, color='tomato', linestyle='--', label='0.70 threshold')
axes[1].set_title('Alphabetic character ratio')
axes[1].set_xlabel('Ratio')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()


#### Sentence Length and Structural Complexity

In [ ]:
nlp_sent = spacy.load('pt_core_news_sm', disable=['ner', 'lemmatizer'])
if 'sentencizer' not in nlp_sent.pipe_names and 'parser' not in nlp_sent.pipe_names:
    nlp_sent.add_pipe('sentencizer')

df_len = df_len.drop(columns=[c for c in ['n_sentences', 'avg_sent_len', 'max_sent_len', 'std_sent_len']
                               if c in df_len.columns])

all_sent_lens = []
doc_n_sents   = []
doc_std_lens  = []

for text in df_len['Sumario']:
    doc  = nlp_sent(text[:10_000])
    lens = [len(s.text.split()) for s in doc.sents]
    all_sent_lens.extend(lens)
    doc_n_sents.append(len(lens))
    doc_std_lens.append(pd.Series(lens).std() if len(lens) > 1 else 0)

df_len['n_sentences']  = doc_n_sents
df_len['std_sent_len'] = doc_std_lens

sent_lens = pd.Series(all_sent_lens, name='words_per_sentence')

print('--- Sentence length distribution (all sentences flattened) ---')
display(sent_lens.describe().round(1).rename({
    'count': 'total sentences', 'mean': 'mean', '50%': 'median',
    '25%': 'p25', '75%': 'p75'
}))
print(f'\n% sentences > 100 words : {(sent_lens > 100).mean()*100:.1f}%')
print(f'% sentences > 200 words : {(sent_lens > 200).mean()*100:.1f}%')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(sent_lens.clip(upper=300).to_numpy(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Sentence length (words) — clipped at 300')
axes[0].set_xlabel('Words per sentence')
axes[0].set_ylabel('Count')

axes[1].hist(df_len['n_sentences'].dropna().to_numpy(), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Sentences per document')
axes[1].set_xlabel('# sentences')
axes[1].set_ylabel('Count')

axes[2].hist(df_len['std_sent_len'].dropna().to_numpy(), bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('Std dev sentence length (within doc)')
axes[2].set_xlabel('Std dev (words)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
complexity = (
    df_len.groupby('TipoDecisao')[['n_sentences', 'std_sent_len']]
    .agg(['mean', 'median'])
    .round(2)
    .sort_values(('n_sentences', 'mean'), ascending=False)
)
print('--- Document structure by TipoDecisao class ---')
display(complexity)


## 5. Decisao Field Analysis

`Decisao` is a secondary outcome field extracted from `Observacoes`.

### 5.1 Coverage and Value Count per TipoDecisao

For each document type: coverage of the `Decisao` field and number of distinct outcome values.

In [ ]:
def parse_decisao(obs):
    if not obs or not isinstance(obs, str):
        return None
    m = re.search(r'Decis[aã]o:\s*(.+)', obs)
    return m.group(1).strip() if m else None

all_data['Decisao'] = all_data['Observacoes'].apply(parse_decisao)

print(f'Records with Decisao : {all_data["Decisao"].notna().sum():,} / {len(all_data):,}')
print(f'Unique Decisao values: {all_data["Decisao"].nunique():,}')

In [ ]:
coverage = (
    all_data.groupby('TipoDecisao')
    .agg(
        total=('Decisao', 'count'),
        with_decisao=('Decisao', lambda x: x.notna().sum()),
        unique_decisao=('Decisao', 'nunique'),
    )
    .assign(
        coverage_pct=lambda d: (d['with_decisao'] / d['total'] * 100).round(1),
    )
)
display(coverage)

In [ ]:
for tipo in all_data['TipoDecisao'].value_counts().index:
    subset = all_data.loc[
        (all_data['TipoDecisao'] == tipo) & all_data['Decisao'].notna(),
        'Decisao'
    ]
    vc = subset.value_counts()
    n_unique = len(vc)
    n_records = len(subset)
    print(f'── {tipo} ── {n_records} records with Decisao | {n_unique} unique values')
    print(vc.head(10).to_string())
    print()

In [ ]:
top_decisao = all_data['Decisao'].value_counts().head(20).index

cross = (
    all_data[all_data['Decisao'].isin(top_decisao)]
    .groupby(['TipoDecisao', 'Decisao'])
    .size()
    .unstack('TipoDecisao', fill_value=0)
    .sort_values(all_data['TipoDecisao'].value_counts().index[0], ascending=False)
)

display(cross)

fig, ax = plt.subplots(figsize=(8, 7))
import numpy as np
im = ax.imshow(cross.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(cross.columns)))
ax.set_xticklabels(cross.columns, rotation=15, ha='right')
ax.set_yticks(range(len(cross.index)))
ax.set_yticklabels([v[:40] + chr(8230) if len(v) > 40 else v for v in cross.index], fontsize=8)
for i in range(len(cross.index)):
    for j in range(len(cross.columns)):
        val = cross.values[i, j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center', fontsize=7,
                    color='white' if val > cross.values.max() * 0.6 else 'black')
plt.colorbar(im, ax=ax, label='Records')
ax.set_title('Top-20 Decisao values by TipoDecisao')
plt.tight_layout()
plt.show()

### 5.2 Full Decisao Value Catalogue per TipoDecisao

All observed `Decisao` values ranked by frequency, per document type.

In [ ]:
for tipo in all_data['TipoDecisao'].value_counts().index:
    vals = (
        all_data.loc[
            (all_data['TipoDecisao'] == tipo) & all_data['Decisao'].notna(),
            'Decisao'
        ]
        .value_counts()
        .rename_axis('Decisao')
        .reset_index(name='count')
        .assign(pct=lambda d: (d['count'] / d['count'].sum() * 100).round(1))
    )
    print(f"── {tipo} ── {len(vals)} unique values")
    display(vals)
    print()

## 6. Decisao by Court Group

Courts are grouped into three families and analysed both together and individually.

In [ ]:
import re, unicodedata
import numpy as np
from scipy.stats import chi2_contingency

_TYPO_FIX = [
    (r'PROVIEMNTO|PROVIMRNTO|PROVIMNTO|PRVIMENTO', 'PROVIMENTO'),
    (r'CONFIRNADA|CONFIRMDA|CONFIMACAO|CONFIRNACAO', 'CONFIRMADA'),
    (r'IMPROEDENTE|IMPR-CEDENTE', 'IMPROCEDENTE'),
    (r'INDEFEERIDA|INDEFEDIDA', 'INDEFERIDA'),
    (r'NAGAD', 'NEGAD'),
    (r'PROCDENTE', 'PROCEDENTE'),
    (r'PARCIALMETE\b|PARCIALMEMTE', 'PARCIALMENTE'),
    (r'CENCEDIDA|CONDEDIDA|CONCEDDO', 'CONCEDIDA'),
]

def _strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def _normalize(raw):
    if not raw or not isinstance(raw, str): return ''
    v = raw.upper().strip()
    v = re.sub(r'[.;,]+$', '', v).strip()
    v = re.sub(r'\s+', ' ', v)
    v = _strip_accents(v)
    for pattern, replacement in _TYPO_FIX:
        v = re.sub(pattern, replacement, v)
    return v

_RULES = [
    ('PARCIALMENTE_PROCEDENTE', re.compile(r'PARCIAL|EM PARTE')),
    ('ANULADA_BAIXA',           re.compile(r'ANULAG?[AO]|ANULAC[AO]O|ANULADO|ANULADA|ANULACAO'
                                            r'|BAIXA|REMESSA|\bREENVIO\b|AMPLIACAO DA MATERIA'
                                            r'|NOVO JULGAMENTO|PROCESSADO ANULADO')),
    ('NAO_CONHECIDO',           re.compile(r'NAO CONHECIMENTO|NAO CONHECIDO|\bNAO ADMITID'
                                            r'|NAO ADMISS\b|INADMISS|\bREJEITAD|\bREJEICAO\b'
                                            r'|INUTILIDADE|\bEXTINTO\b|NAO SE TOMA CONHECIMENTO'
                                            r'|NAO TOMAR CONHECIMENTO|NAO RECONHECIMENTO')),
    ('IMPROCEDENTE',            re.compile(r'IMPROCEDENTE|IMPROCEDENCIA|NEGADO PROVIMENTO|NAO PROVIDO'
                                            r'|NEGAR PROVIMENTO|NAO PROCEDENTE|NEGADA|NEGADO|NEGA-SE'
                                            r'|DESATENDID|REVISTA IMPROCEDENTE|RECLAMACAO INDEFERIDA'
                                            r'|INDEFERIDA A RECLAMACAO')),
    ('PROCEDENTE',              re.compile(r'PROCEDENTE|PROCEDENCIA|PROVIDO|PROVIDA|CONCEDID'
                                            r'|PROVIMENTO|DEFERIMENT|DEFERIDA\b|DEFERIDO\b'
                                            r'|ADMITIDA A REVISTA|CONCEDER PROVIMENTO|DAR PROVIMENTO')),
    ('CONFIRMADA',              re.compile(r'CONFIRMAD|CONFIRMAC')),
    ('REVOGADA',                re.compile(r'REVOGAD|REVOGAC')),
    ('ALTERADA',                re.compile(r'ALTERAD|MODIFICAD')),
]

def categorize_le(raw):
    v = _normalize(raw)
    if not v: return 'OUTRO'
    for label, rx in _RULES:
        if rx.search(v): return label
    return 'OUTRO'

all_data['Decisao_le'] = all_data['Decisao'].apply(categorize_le)
print('Decisao_le applied.')
print(all_data['Decisao_le'].value_counts())

### 6.1 Coverage per Court and per Group

In [ ]:
GROUPS = {
    'STJ/STA':   ['STJ', 'STA'],
    'TCAN/TCAS': ['TCAN', 'TCAS'],
    'TRx':       ['TRG', 'TRE', 'TRL', 'TRP', 'TRC'],
}

def assign_group(src):
    for g, members in GROUPS.items():
        if src in members: return g
    return 'OTHER'

all_data['group'] = all_data['source'].apply(assign_group)

in_groups = all_data[all_data['group'] != 'OTHER'].copy()

cov_src = (
    in_groups.groupby('source')
    .apply(lambda g: pd.Series({
        'n_total':        len(g),
        'n_with_decisao': g['Decisao'].notna().sum(),
        'pct_coverage':   round(g['Decisao'].notna().mean() * 100, 1),
    }))
    .reset_index()
    .sort_values('source')
)

cov_grp = (
    in_groups.groupby('group')
    .apply(lambda g: pd.Series({
        'n_total':        len(g),
        'n_with_decisao': g['Decisao'].notna().sum(),
        'pct_coverage':   round(g['Decisao'].notna().mean() * 100, 1),
    }))
    .reset_index()
)

print('Coverage per source:')
display(cov_src)
print('\nCoverage per group:')
display(cov_grp)

### 6.2 Decisao_le Distribution — Per Court

In [ ]:
cat_order = ['PROCEDENTE', 'PARCIALMENTE_PROCEDENTE', 'IMPROCEDENTE',
             'NAO_CONHECIDO', 'ANULADA_BAIXA', 'CONFIRMADA', 'REVOGADA', 'ALTERADA', 'OUTRO']

labeled = in_groups[in_groups['Decisao'].notna()].copy()

dist_src = (
    labeled.groupby(['source', 'Decisao_le'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=cat_order, fill_value=0)
)
dist_src_pct = dist_src.div(dist_src.sum(axis=1), axis=0).mul(100).round(1)

print('Absolute counts:')
display(dist_src)
print('\nRow percentages (%):')
display(dist_src_pct)


### 6.3 Decisao_le Distribution — Per Group

In [ ]:
dist_grp = (
    labeled.groupby(['group', 'Decisao_le'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=cat_order, fill_value=0)
)
dist_grp_pct = dist_grp.div(dist_grp.sum(axis=1), axis=0).mul(100).round(1)

print('Absolute counts:')
display(dist_grp)
print('\nRow percentages (%):')
display(dist_grp_pct)


### 6.4 Heatmap — Decisao_le % by Court and by Group

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [3, 1]})

sns.heatmap(
    dist_src_pct,
    ax=axes[0],
    annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': '%'},
    vmin=0, vmax=70
)
axes[0].set_title('Decisao_le % by Court', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Court')
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(
    dist_grp_pct,
    ax=axes[1],
    annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': '%'},
    vmin=0, vmax=70
)
axes[1].set_title('Decisao_le % by Group', fontsize=12)
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Decisao class distribution by court and group', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


### 6.5 Stacked Bar — Proportional Decisao Distribution

In [ ]:
COLOR_MAP = {
    'PROCEDENTE':               '#2ecc71',
    'PARCIALMENTE_PROCEDENTE':  '#f39c12',
    'IMPROCEDENTE':             '#e74c3c',
    'NAO_CONHECIDO':            '#9b59b6',
    'ANULADA_BAIXA':            '#3498db',
    'CONFIRMADA':               '#1abc9c',
    'REVOGADA':                 '#e67e22',
    'ALTERADA':                 '#95a5a6',
    'OUTRO':                    '#bdc3c7',
}
colors = [COLOR_MAP[c] for c in cat_order]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

dist_src_pct.plot(kind='bar', stacked=True, color=colors, ax=axes[0], width=0.75)
axes[0].set_title('Decisao_le (%) — per court')
axes[0].set_xlabel('Court')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(loc='upper right', fontsize=8, title='Class')
axes[0].set_ylim(0, 110)

dist_grp_pct.plot(kind='bar', stacked=True, color=colors, ax=axes[1], width=0.5)
axes[1].set_title('Decisao_le (%) — per group')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=10)
axes[1].legend(loc='upper right', fontsize=8, title='Class')
axes[1].set_ylim(0, 110)

plt.tight_layout()
plt.show()


### 6.6 Chi-Squared Test — Association between Court and Decisao

In [ ]:
non_outro = labeled[labeled['Decisao_le'] != 'OUTRO'].copy()

ct_src = pd.crosstab(non_outro['source'], non_outro['Decisao_le'])
chi2_s, p_s, dof_s, _ = chi2_contingency(ct_src)
n_s = ct_src.values.sum()
cramers_v_s = np.sqrt(chi2_s / (n_s * (min(ct_src.shape) - 1)))

ct_grp = pd.crosstab(non_outro['group'], non_outro['Decisao_le'])
chi2_g, p_g, dof_g, _ = chi2_contingency(ct_grp)
n_g = ct_grp.values.sum()
cramers_v_g = np.sqrt(chi2_g / (n_g * (min(ct_grp.shape) - 1)))

stats = pd.DataFrame([
    {'level': 'per source', 'chi2': round(chi2_s, 1), 'dof': dof_s,
     'p_value': f'{p_s:.2e}', "Cramér's V": round(cramers_v_s, 4)},
    {'level': 'per group',  'chi2': round(chi2_g, 1), 'dof': dof_g,
     'p_value': f'{p_g:.2e}', "Cramér's V": round(cramers_v_g, 4)},
])
display(stats)

print("\nCramér's V interpretation: 0 = no association, 1 = perfect association")
print(f"Source-level V = {cramers_v_s:.3f}  →  moderate-to-strong")
print(f"Group-level  V = {cramers_v_g:.3f}  →  moderate")

### 6.7 Individual Court Signatures (Top Raw Values + Category Mix)

In [ ]:
all_sources = ['STJ', 'STA', 'TCAN', 'TCAS', 'TRG', 'TRE', 'TRL', 'TRP', 'TRC']

for src in all_sources:
    sub = labeled[labeled['source'] == src]
    if len(sub) == 0:
        print(f'{src}: NO LABELED DATA')
        print(f'  → Observacoes does not contain the Decisão: label.')
        print(f'  → TCAS uses Votação: and other fields instead.')
        print()
        continue

    vc_raw = sub['Decisao'].value_counts().head(8)
    vc_le  = sub['Decisao_le'].value_counts()
    dom    = vc_le.index[0]
    dom_pct = vc_le.iloc[0] / len(sub) * 100

    print(f'─── {src}  (n={len(sub):,} labeled | dominant: {dom} {dom_pct:.1f}%) ───')
    cat_line = '  '.join(f'{k}:{v}({v/len(sub)*100:.0f}%)' for k, v in vc_le.items())
    print(f'  Categories: {cat_line}')
    print(f'  Top-8 raw values:')
    for val, cnt in vc_raw.items():
        print(f'    {cnt:5d}  {val}')
    print()


### 6.8 Key Findings